# Efficient de-reservation in matching markets

This notebook reproduces the revised evaluation. The Indian design uses expected
SC and ST applicant shares of 16.6% and 8.6%. It sweeps expected OBC shares of 2%,
10%, 27%, 42%, and 50%. The 42% setting also anchors the capacity, preference
list length, and merit score sweeps. The study evaluates six mechanisms on 100
paired market draws per setting across 14 Indian and 18 overlapping settings.

A fresh clone reuses bundled observations for the overlapping study, the MD
runtime benchmark, and the fixed examples. The notebook can generate each of
these from the same design and seeds when the saved data are missing.

## 1. Setup

Use the Python environment from the [replication guide](../README.md), then
open this notebook from the repository root.

    python -m pip install -r dereservation_study/requirements.txt
    jupyter lab dereservation_study/experiments.ipynb

Run all cells with `RESUME = True`. Each fresh market is checkpointed under
`dereservation_study/results/notebook_indian_demographics/`. Completed studies
are loaded when the notebook runs again. Reports and figures are written to
the `reports/` subfolder.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'dereservation_study' / 'generation.py').is_file()
             and (p / 'src' / 'instance_generation.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from inside the repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from dereservation_study.algorithms import algorithm_suite
from dereservation_study.analysis import (
    SWEEP_METRICS, audit_results, benchmark_weighted_engines, plot_sweep,
    runtime_statistics, save_figure,
)
from dereservation_study.examples import bt_avoidable_vacancies, compliance_tradeoff
from dereservation_study.experiments import (
    METRICS, StudyResults, paired_differences, proposed_design, summarize,
)
from dereservation_study.generation import generate_overlapping_market
from dereservation_study.metrics import evaluate_outcome
from dereservation_study.policies import compliance_frontier
from dereservation_study.reporting import (
    PAPER_ARCHIVE, TABLE_NAMES, load_or_run_study, load_runtime_cache,
)

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
BASE_SEED = 271828
RESUME = True
PLOT_ALL_SWEEPS = True
OBC_SHARES = (.02, .10, .27, .42, .50)
INDIAN_BASELINE = {'obc_population': .42, 'sc_population': .166, 'st_population': .086}

OUTPUT_DIR = ROOT / 'dereservation_study' / 'results' / 'notebook_indian_demographics'
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'

archived_scenarios, repetitions = proposed_design('paper')
obc_template = next(s for s in archived_scenarios
                    if s.regime == 'indian' and s.axis == 'obc_population')
indian_scenarios = [
    replace(obc_template, name=f'indian_obc_population={share}', value=share,
            parameters={**obc_template.parameters, **INDIAN_BASELINE, 'obc_population': share})
    for share in OBC_SHARES
]
indian_scenarios.extend(
    replace(s, parameters={**s.parameters, **INDIAN_BASELINE})
    for s in archived_scenarios
    if s.regime == 'indian' and s.axis != 'obc_population'
)
overlapping_scenarios = [s for s in archived_scenarios if s.regime == 'overlapping']
suites = {regime: algorithm_suite(regime, lambdas=()) for regime in ('indian', 'overlapping')}

print(f'{len(indian_scenarios)} Indian and {len(overlapping_scenarios)} overlapping settings, '
      f'{repetitions} paired draws per setting')
display(pd.DataFrame([{'factor': s.axis, 'value': s.value} for s in indian_scenarios])
        .groupby('factor').value.agg(list))

def saved_table(name, generate, read=pd.read_csv):
    local = OUTPUT_DIR / name
    if RESUME:
        for candidate in (local, PAPER_ARCHIVE / name):
            if candidate.is_file():
                return read(candidate)
    table = generate()
    local.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(local, index=False)
    return table

### Mechanisms and design

- **Indian vertical reservations.** Fixed-quota DA fills open seats first, then
  reserved seats. Multi-run DA converts unused OBC quotas between complete DA
  runs. BT handles conversions within the school's choice step. SC and ST seats
  remain restricted.
- **Overlapping eligibility.** DA-open uses school priorities and aggregate
  capacity. Smart-Hard-Reserves checks whether a cohort can fit the original
  open and reserved seats. Soft-Reserves applies demographic floors at
  rejection. One student can count toward several floors.
- **Both settings.** MD maximizes local headcount then category-priority
  claims. RP puts original reserved-seat use between these two objectives.
  Merit maximizes headcount then the admitted cohort's total merit score.

Each draw gives all six mechanisms the same market. The Indian OBC sweep
holds the other expected applicant shares fixed at the SC and ST population
shares from the [2011 Census](https://censusindia.gov.in/nada/index.php/catalog/42604/download/46256/Census%20of%20India%202011-PCA%20Release.pdf).
The auxiliary sweeps vary one factor at a time at the 42% OBC baseline, close
to the 41.7% population estimate in the [2009-10 National Sample Survey](https://mospi.gov.in/sites/default/files/publication_reports/nss_rep_563_13mar15.pdf).

## 2. Reproduce the manuscript's avoidable-vacancy example

Example 2.1 has eight students and eight positions. BT and multi-run DA admit six;
MD can move admitted SC/ST students to their dedicated positions and admit all eight.
Original physical seat labels are retained throughout the comparison.

In [ ]:
def compute_vacancy_example():
    toy = bt_avoidable_vacancies()
    rows = []
    for algorithm in suites['indian']:
        outcome = algorithm.run(toy)
        metrics = evaluate_outcome(toy, outcome, algorithm, runtime=0)
        rows.append({
            'algorithm': algorithm.name,
            'matched': metrics['num_matches'],
            'original_reserved': metrics['original_reserved_seats'],
            'ED_seats': metrics['dereserved_seats'],
            'choice_blocks': metrics['blocking_pairs'],
            'assignment': str(outcome.assignments[0]),
        })
    return pd.DataFrame(rows)

example_table = saved_table('paper_example_2_1.csv', compute_vacancy_example)
display(example_table)
assert example_table.set_index('algorithm').loc['BT', 'matched'] == 6
assert example_table.set_index('algorithm').loc['MD', 'matched'] == 8

## 3. Run or load the two market studies

The Indian and overlapping studies use their own checkpoints. The notebook
can also load the overlapping rows from the bundled paper archive. It checks
the archived design, seeds, and algorithm definitions before using those rows.
The revised Indian design uses its own results.


An original reserved position filled by an originally eligible student contributes
to $D$. A reserved position filled by an additional eligible student contributes
to $T$. Together, $D+T$ counts the occupied original reserved positions. BT and
multi-run DA also record which OBC quotas were converted administratively.

In [ ]:
def has_study_tables(path):
    return (path / 'manifest.json').is_file() and all(
        (path / f'{name}.csv').is_file() or (path / f'{name}.csv.gz').is_file()
        for name in TABLE_NAMES
    )

overlap_output = OUTPUT_DIR / 'overlapping'
if RESUME and not has_study_tables(overlap_output) and has_study_tables(PAPER_ARCHIVE):
    archived_study = load_or_run_study(
        archived_scenarios, repetitions, BASE_SEED, suites,
        output_dir=PAPER_ARCHIVE, resume=True,
    )
    overlap_tables = {}
    for name in TABLE_NAMES:
        frame = getattr(archived_study, name)
        overlap_tables[name] = frame.loc[frame.regime.eq('overlapping')].reset_index(drop=True)
    overlapping_study = StudyResults(**overlap_tables, manifest=archived_study.manifest)
    print(f'Reused {len(overlapping_study.outcomes):,} archived overlapping outcomes.')
else:
    overlapping_study = load_or_run_study(
        overlapping_scenarios, repetitions, BASE_SEED, {'overlapping': suites['overlapping']},
        output_dir=overlap_output, resume=RESUME,
    )

indian_study = load_or_run_study(
    indian_scenarios, repetitions, BASE_SEED, {'indian': suites['indian']},
    output_dir=OUTPUT_DIR / 'indian', resume=RESUME,
)

tables = {
    name: pd.concat([getattr(indian_study, name), getattr(overlapping_study, name)], ignore_index=True)
    for name in TABLE_NAMES
}
study = StudyResults(**tables, manifest={
    'indian': indian_study.manifest, 'overlapping': overlapping_study.manifest,
})
REPORT_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([audit_results(study)]))
feasible = study.outcomes.loc[study.outcomes.eligibility_feasible]
outcome_metrics = [metric for metric in METRICS if metric != 'runtime']

### Domain and label checks before interpreting comparisons

The original soft-bound code can return an unsupported cohort when new transfer
permissions are restricted. Such outputs remain in the raw file, but default
summaries and paired differences exclude them. Undefined seat metrics are missing,
not zero. Full transfer access in the default overlap study makes these benchmark
cohorts implementable.

Earlier backends report placements only. Their `posthoc_MD_support` seat labels
are a deterministic reporting convention. `original_reserved_seats_max_for_cohort`
also measures the greatest D for the same students; this reveals label sensitivity.
The general benchmarks use their existing school priorities and soft-bound
interpretations; they are not relabeled as exact implementations of the new model.

In [ ]:
domain = (study.outcomes.groupby(['regime', 'axis', 'value', 'algorithm'], as_index=False)
          .agg(runs=('eligibility_feasible', 'size'),
               feasible_runs=('eligibility_feasible', 'sum')))
domain['unsupported_runs'] = domain.runs - domain.feasible_runs
display(domain.loc[domain.unsupported_runs.gt(0)])
domain.to_csv(REPORT_DIR / 'domain_checks.csv', index=False)

summary = summarize(study.outcomes, outcome_metrics)
summary.to_csv(REPORT_DIR / 'metric_summary.csv', index=False)
runtime_statistics(feasible, ['regime', 'scenario', 'axis', 'value', 'algorithm']).to_csv(
    REPORT_DIR / 'algorithm_runtime.csv', index=False,
)

headline = feasible.loc[
    feasible.regime.eq('indian') & feasible.axis.eq('obc_population')
    & feasible.value.eq(INDIAN_BASELINE['obc_population'])
]
headline_table = headline.groupby('algorithm').agg(
    markets=('instance_id', 'size'),
    matched=('num_matches', 'mean'),
    mean_rank=('average_rank', 'mean'),
    original_reserved=('original_reserved_seats', 'mean'),
    transfers=('dereserved_seats', 'mean'),
    protected=('minority_matched', 'mean'),
    median_seconds=('runtime', 'median'),
).reindex([algorithm.name for algorithm in suites['indian']])
headline_table.to_csv(REPORT_DIR / 'indian_42_summary.csv')
display(headline_table.round(3))

## 4. Welfare, seat use, envy, waste, and representation

Algorithms share each metric's axes. Wider strokes and distinct dash patterns
show overlaid curves. Symbols between measured settings identify exactly
coincident segments. Values at measured settings and error bars keep their
original coordinates. Gaps indicate settings with no feasible observations.

Plots show means and approximate 95% confidence intervals across draws.
Runtime shows the median and interquartile range. The raw timing series
include isolated long observations.

The average rank includes students who remain unmatched at rank one plus
their list length. Protected shares count each student once, including
students eligible in several categories.

In [ ]:
main_sweeps = {('indian', 'obc_population'), ('overlapping', 'overlap')}
sweeps = study.outcomes[['regime', 'axis']].drop_duplicates().itertuples(index=False, name=None)
for regime, axis in sweeps:
    if not PLOT_ALL_SWEEPS and (regime, axis) not in main_sweeps:
        continue
    for metrics in SWEEP_METRICS.values():
        fig = plot_sweep(study.outcomes, regime, axis, metrics, output_dir=FIGURE_DIR)
        if (regime, axis) in main_sweeps:
            display(fig)
        plt.close(fig)

### Three blocking diagnostics

`blocking_pairs` uses the paper's test: a student prefers the school and its
announced choice accepts her from its current cohort plus that student. Category
exchanges are allowed. `choice_waste_pairs` increases headcount; `choice_envy_pairs`
is the replacement case. These two counts partition policy blocking.

`category_envy_pairs` and `category_waste_pairs` inspect the reported category
assignment without exchanges: a lower-priority occupant or an eligible vacancy.
They may overlap; their union is saved separately. `merit_*` diagnostics ignore
reservation protections. All count student-school edges, not student-student
triples or numbers of vacant seats. Use `empty_seats` for the latter.

Multi-run DA is tested under frozen final quotas. A zero policy-blocking count
does not imply zero ordinary merit envy. The legacy overlapping soft-bound fallback
is checked empirically, without assuming a general substitutes theorem.

In [ ]:
stability_columns = ['blocking_pairs', 'choice_envy_pairs', 'choice_waste_pairs',
                     'category_envy_pairs', 'category_waste_pairs', 'category_blocking_pairs',
                     'merit_envy_pairs', 'merit_waste_pairs', 'merit_blocking_pairs']
display(feasible.groupby(['regime', 'algorithm'])[stability_columns].mean().round(3))
assert (study.outcomes.blocking_pairs ==
        study.outcomes.choice_envy_pairs + study.outcomes.choice_waste_pairs).all()

## 5. Paired comparisons and student welfare relative to BT

Differences are algorithm minus reference on the same instance. Negative rank
differences and positive matching differences favor the algorithm. Uncertainty
is computed on paired differences, not by combining separate algorithm error bars.
Student-level welfare is a separate statistic: students better, equal, or worse
relative to BT in each Indian market. The Indian MD-versus-BT comparison is an
empirical consistency check in the theorem's domain, not a guarantee for other
weighted policies or general overlapping markets.

In [ ]:
paired_bt = paired_differences(study.outcomes, 'BT', outcome_metrics)
paired_soft = paired_differences(study.outcomes, 'Soft-Reserves', outcome_metrics)
paired_bt.to_csv(REPORT_DIR / 'paired_vs_BT.csv', index=False)
paired_soft.to_csv(REPORT_DIR / 'paired_vs_Soft-Reserves.csv', index=False)
display(paired_bt[(paired_bt.algorithm == 'MD') &
                  paired_bt.metric.isin(['average_rank', 'num_matches', 'original_reserved_seats'])]
        [['scenario', 'metric', 'n', 'mean', 'ci95_low', 'ci95_high']])

welfare = study.welfare[(study.welfare.regime == 'indian') &
                        (study.welfare.algorithm == 'MD') &
                        study.welfare.eligibility_feasible]

counts = (welfare.loc[welfare.axis.eq('obc_population')]
          .groupby('value')[['students_better', 'students_equal', 'students_worse']].sum()
          .rename_axis('obc_share'))
counts.to_csv(REPORT_DIR / 'indian_md_vs_bt_counts.csv')
display(counts)

by_supply = counts.div(counts.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7, 4))
by_supply.plot.bar(stacked=True, ax=ax, rot=0)
ax.set_xticks(range(len(by_supply)), labels=[f'{share:.0%}' for share in by_supply.index])
ax.set(xlabel='Expected OBC applicant share', ylabel='Share of students',
       title='MD student ranks relative to BT (paired synthetic instances)')
ax.legend(['Better', 'Equal', 'Worse'], loc='upper right')
save_figure(fig, FIGURE_DIR, 'indian_md_vs_bt_student_welfare')
display(fig)
plt.close(fig)

### Per-group representation

Per-type populations and counts can overlap. ANY_PROTECTED counts the union
and UNPROTECTED counts its complement. The table selects 42% OBC in the
Indian market and 0.5 overlap in the overlapping market.

In [ ]:
selected = feasible[
    ((feasible.axis == 'obc_population') & (feasible.value == INDIAN_BASELINE['obc_population'])) |
    ((feasible.axis == 'overlap') & (feasible.value == 0.5))
][['instance_id', 'algorithm']]

group_table = (study.groups.merge(selected, on=['instance_id', 'algorithm'], validate='many_to_one')
               .groupby(['scenario', 'algorithm', 'group'], as_index=False)
               .agg(mean_population=('group_population', 'mean'),
                    mean_matched=('group_matched', 'mean'),
                    admission_rate=('group_admission_rate', 'mean'),
                    share_of_matched=('group_share_matched', 'mean'),
                    average_rank=('group_average_rank', 'mean')))
display(group_table[group_table.group == 'ANY_PROTECTED'])
group_table.to_csv(REPORT_DIR / 'group_summary_selected_scenarios.csv', index=False)

## 6. Exact local original-seat compliance frontier

Fix one institution and one applicant pool. For each feasible headcount k,
D*(k) is the greatest possible original reserved-seat compliance. Example 6.4
admits three students with D=1, or two with D=2. This is a local descriptive
frontier, separate from market-wide stable matching outcomes. Saved frontier
tables are reused without recomputing flow solutions.

In [ ]:
frontier = saved_table(
    'example_6_4_local_frontier.csv',
    lambda: pd.DataFrame(compliance_frontier(compliance_tradeoff(), 0)).drop(columns='assignment'),
)
display(frontier)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(frontier.headcount, frontier.original_reserved_seats, 'o-', label='Exact D*(k)')
ax.set(xlabel='Admissions at the fixed institution', ylabel='Original reserved seats filled by E0',
       title='Manuscript Example 6.4: local compliance frontier')
save_figure(fig, FIGURE_DIR, 'example_6_4_frontier')
display(fig)
plt.close(fig)


def compute_generated_frontier():
    market = generate_overlapping_market(
        seed=BASE_SEED, num_students=24, num_institutions=3,
        reserve_fraction=.8, transferable_fraction=.5, transfer_access=.5, priority_noise=.5,
    )
    return pd.DataFrame(compliance_frontier(market, 0)).drop(columns='assignment')

generated_frontier = saved_table('generated_local_frontier.csv', compute_generated_frontier)
display(generated_frontier)

## 7. Runtime of the two exact MD implementations

The benchmark checks that incremental alternating-path updates and recomputed
minimum-cost flows return identical assignments. Plot points are medians and
error bars show the middle half of the recorded times. Both axes are logarithmic.
The paper profile comes with 20 paired trials at each size.

In [ ]:
sizes = (100, 250, 500, 1000, 2000)
runtime_repetitions = 20
runtime_schools = 20
runtime_data = saved_table(
    'runtime_engines.csv',
    lambda: benchmark_weighted_engines(
        sizes, runtime_repetitions, num_institutions=runtime_schools,
    ),
    lambda path: load_runtime_cache(path, sizes, runtime_repetitions, runtime_schools),
)
runtime_summary = runtime_statistics(runtime_data, ['engine', 'num_students'])
display(runtime_summary)
runtime_summary.to_csv(REPORT_DIR / 'runtime_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
styles = {'flow': ('Recomputed flow', '#2865a8', '-', 's', 3.0),
          'incremental': ('Alternating path', '#007f66', '--', 'o', 1.7)}
for engine, group in runtime_summary.groupby('engine'):
    label, color, linestyle, marker, width = styles[engine]
    ax.errorbar(group.num_students, group['median'],
                yerr=[group['median'] - group.q1, group.q3 - group['median']],
                color=color, linestyle=linestyle, linewidth=width, marker=marker,
                markersize=5, capsize=3, label=label)
ax.set(xlabel='Students', ylabel='Median DA runtime (seconds)', xscale='log', yscale='log')
ax.set_xticks(sizes, labels=[str(size) for size in sizes])
ax.legend(frameon=False)
save_figure(fig, FIGURE_DIR, 'runtime_engines')
display(fig)
plt.close(fig)

## 8. Reports

The reports include the 42% OBC summary, the MD-versus-BT student counts
at each OBC share, and the figures for all Indian and overlapping sweeps.
The revised Indian observations and their manifest are in the run's Indian
subfolder.

In [ ]:
print('Report directory:', REPORT_DIR)
display(pd.DataFrame({'table': sorted(p.name for p in REPORT_DIR.glob('*.csv'))}))